# M8C3 — Data Gate Self-Check

Use this notebook **after** you have prepared your repository evidence.

It checks:
- repository files and required sections;
- generic dataset risks;
- target balance and missingness;
- optional train/validation overlap, class ratio, group and time boundaries;
- project-type reminders;
- a suggested Green / Yellow / Red status.

> **Important:** no notebook can prove that every feature is leakage-safe. Feature timing and real-use availability still require human judgment.


> FontSense copy: teacher-provided validator logic retained. Only the PROJECT configuration below is set to the real repository manifest schema.

## Step 0 — Configure your project

Use `MODE = "DEMO"` first to see a complete working example.  
Then switch to `MODE = "PROJECT"` and update the paths.

Accepted project types:
- `tabular`
- `nlp`
- `cv`
- `unsupervised`
- `time_group`


In [ ]:
from pathlib import Path
import json
import re
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

MODE = "PROJECT"

PROJECT_TYPE = "cv"
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Run this notebook from the FontSense repository root or notebooks folder.")

DATASET_PATH = PROJECT_ROOT / "reports" / "dataset" / "full_manifest.csv"
TARGET_COLUMN = "category"
GROUP_COLUMN = "family"
GROUP_OVERLAP_FORBIDDEN = True
TIME_COLUMN = None
TEXT_COLUMN = None
SPLIT_COLUMN = "split"

TRAIN_PATH = None
VALID_PATH = None

MAX_PLOT_CATEGORIES = 15
print("Configuration loaded for FontSense.")
print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)


## Step 1 — Create a safe demo project or resolve your project paths

In [ ]:
def create_demo_project(root: Path) -> dict:
    rng = np.random.default_rng(42)
    root.mkdir(parents=True, exist_ok=True)
    (root / "data").mkdir(exist_ok=True)
    (root / "docs").mkdir(exist_ok=True)
    (root / "reports").mkdir(exist_ok=True)

    n = 900
    dates = pd.date_range("2025-01-01", periods=300, freq="D")
    df = pd.DataFrame({
        "customer_id": rng.integers(1000, 1260, n),
        "event_date": rng.choice(dates, n),
        "age": rng.normal(36, 11, n).round(1),
        "monthly_spend": rng.lognormal(4.0, 0.45, n).round(2),
        "contract_type": rng.choice(["monthly", "annual", "two_year"], n, p=[0.55, 0.30, 0.15]),
        "support_calls": rng.poisson(1.3, n),
    })
    logits = -1.6 + 0.45*(df["contract_type"] == "monthly") + 0.22*df["support_calls"] - 0.006*df["monthly_spend"]
    p = 1 / (1 + np.exp(-logits))
    df["target"] = rng.binomial(1, p)
    df.loc[rng.choice(df.index, 38, replace=False), "age"] = np.nan

    split_date = pd.Timestamp("2025-08-15")
    df["split"] = np.where(pd.to_datetime(df["event_date"]) < split_date, "train", "validation")
    df.to_csv(root / "data" / "project_data.csv", index=False)

    (root / "data" / "README.md").write_text(
        "# Data README\n\n## Dataset identity\nDemo data generated inside this notebook.\n"
        "## What the data represents\nOne row is a customer event. Target is a binary event outcome.\n"
        "## Known limitations\nSynthetic demonstration only.\n", encoding="utf-8")
    (root / "docs" / "data_audit.md").write_text(
        "# Data Audit and Leakage-Safe Pipeline\n\n## Audit summary\nDemo audit.\n"
        "## Data-quality issue log\nMissing age values.\n"
        "## Split decision\nChronological split.\n"
        "## Leakage risks and controls\nPreprocessing fit on train only.\n"
        "## Preprocessing design\nReusable pipeline planned.\n"
        "## Data Gate status\nYellow until final preprocessing commit.\n", encoding="utf-8")
    (root / "PROJECT_STATUS.md").write_text(
        "# Project Status\n\n## Current stage\nStage: Data Audit\nData Gate status: Yellow\n"
        "## Evidence map\ndata/README.md\ndocs/data_audit.md\n"
        "## Current correction or blocker\nFinish reusable preprocessing.\n", encoding="utf-8")
    return {
        "root": root,
        "dataset": root / "data" / "project_data.csv",
        "target": "target",
        "group": "customer_id",
        "time": "event_date",
        "split": "split",
    }

if MODE.upper() == "DEMO":
    demo_root = Path("/content/m8c3_data_gate_demo") if Path("/content").exists() else Path.cwd() / "_m8c3_data_gate_demo"
    demo = create_demo_project(demo_root)
    PROJECT_ROOT = demo["root"]
    DATASET_PATH = demo["dataset"]
    TARGET_COLUMN = demo["target"]
    GROUP_COLUMN = demo["group"]
    TIME_COLUMN = demo["time"]
    SPLIT_COLUMN = demo["split"]

PROJECT_ROOT = Path(PROJECT_ROOT)
DATASET_PATH = Path(DATASET_PATH) if DATASET_PATH else None
TRAIN_PATH = Path(TRAIN_PATH) if TRAIN_PATH else None
VALID_PATH = Path(VALID_PATH) if VALID_PATH else None

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATASET_PATH)


## Step 2 — Repository evidence checks

In [ ]:
results = []

def record(status, check, evidence, action=""):
    results.append({
        "status": status,
        "check": check,
        "evidence": str(evidence),
        "required_action": action
    })

required_files = {
    "Data source file": PROJECT_ROOT / "data" / "README.md",
    "Data audit": PROJECT_ROOT / "docs" / "data_audit.md",
    "Project status": PROJECT_ROOT / "PROJECT_STATUS.md",
}

for label, file_path in required_files.items():
    if file_path.exists():
        record("PASS", label, file_path)
    else:
        record("FAIL", label, file_path, f"Create {file_path.relative_to(PROJECT_ROOT)}")

section_checks = {
    PROJECT_ROOT / "data" / "README.md": ["source", "limitation"],
    PROJECT_ROOT / "docs" / "data_audit.md": ["audit", "split", "leakage", "preprocessing", "status"],
    PROJECT_ROOT / "PROJECT_STATUS.md": ["stage", "status", "evidence"],
}

for file_path, keywords in section_checks.items():
    if not file_path.exists():
        continue
    text = file_path.read_text(encoding="utf-8", errors="ignore").lower()
    for keyword in keywords:
        if keyword in text:
            record("PASS", f"{file_path.name}: contains '{keyword}'", file_path)
        else:
            record("WARN", f"{file_path.name}: contains '{keyword}'", file_path, f"Add a clear {keyword} section")

display(pd.DataFrame(results))


## Step 3 — Load and audit the dataset

In [ ]:
if DATASET_PATH is None or not DATASET_PATH.exists():
    raise FileNotFoundError(
        "Set DATASET_PATH to a CSV file that exists, or run MODE='DEMO' first."
    )

df = pd.read_csv(DATASET_PATH)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
display(df.head())
display(pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2),
    "unique": [df[c].nunique(dropna=False) for c in df.columns],
}).sort_values("missing_pct", ascending=False))


In [ ]:
duplicate_rows = int(df.duplicated().sum())
constant_columns = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
identifier_like = [
    c for c in df.columns
    if c.lower() == "id" or c.lower().endswith("_id") or c.lower().startswith("id_")
]

record("PASS" if duplicate_rows == 0 else "WARN",
       "Exact duplicate rows",
       duplicate_rows,
       "Investigate whether duplicates are errors, repeated events, or group leakage.")

record("PASS" if not constant_columns else "WARN",
       "Constant columns",
       constant_columns,
       "Remove or justify constant columns.")

record("MANUAL" if identifier_like else "PASS",
       "Identifier-like columns",
       identifier_like,
       "Check whether each ID should be excluded from features or used for grouping.")

print("Duplicate rows:", duplicate_rows)
print("Constant columns:", constant_columns)
print("Identifier-like columns:", identifier_like)


### Visual check — missingness

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0].head(20)

if len(missing):
    plt.figure(figsize=(10, 4.5))
    missing.sort_values().plot(kind="barh")
    plt.title("Top missing-value percentages")
    plt.xlabel("Missing values (%)")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")


### Target or objective check

In [ ]:
if PROJECT_TYPE == "unsupervised" or TARGET_COLUMN is None:
    record("MANUAL", "Unsupervised objective", "No supervised target expected",
           "Document feature rationale and the stability/usefulness evaluation plan.")
    print("Unsupervised route: no target balance plot is required.")
elif TARGET_COLUMN not in df.columns:
    record("FAIL", "Target column exists", TARGET_COLUMN, "Correct TARGET_COLUMN or dataset schema.")
    print("Target column not found:", TARGET_COLUMN)
else:
    target_counts = df[TARGET_COLUMN].value_counts(dropna=False)
    target_share = (df[TARGET_COLUMN].value_counts(normalize=True, dropna=False) * 100).round(2)
    target_table = pd.DataFrame({"count": target_counts, "percent": target_share})
    display(target_table)

    plt.figure(figsize=(7, 4))
    target_counts.head(MAX_PLOT_CATEGORIES).plot(kind="bar")
    plt.title(f"Target distribution: {TARGET_COLUMN}")
    plt.xlabel("Target value")
    plt.ylabel("Rows")
    plt.tight_layout()
    plt.show()

    if len(target_share) == 2 and target_share.min() < 15:
        record("WARN", "Target imbalance", target_share.to_dict(),
               "Preserve ratios; justify metrics; keep any resampling inside training only.")
    else:
        record("PASS", "Target distribution inspected", target_share.to_dict())


## Step 4 — Split-boundary checks

In [ ]:
def check_split_frames(train_df, valid_df):
    print("Train shape:", train_df.shape, "| Validation shape:", valid_df.shape)
    if TARGET_COLUMN and TARGET_COLUMN in train_df and TARGET_COLUMN in valid_df:
        ratios = pd.DataFrame({
            "train": train_df[TARGET_COLUMN].value_counts(normalize=True),
            "validation": valid_df[TARGET_COLUMN].value_counts(normalize=True)
        }).fillna(0)
        display((ratios * 100).round(2))
        record("PASS", "Train/validation target ratios", ratios.to_dict())

    if GROUP_COLUMN and GROUP_COLUMN in train_df and GROUP_COLUMN in valid_df:
        overlap = set(train_df[GROUP_COLUMN].dropna()).intersection(valid_df[GROUP_COLUMN].dropna())
        if len(overlap) == 0:
            status = "PASS"
            action = ""
        elif GROUP_OVERLAP_FORBIDDEN:
            status = "FAIL"
            action = "Use a group-aware split; the same entity must not cross the boundary."
        else:
            status = "MANUAL"
            action = "Confirm that repeated groups are expected in real future use; otherwise use a group-aware split."
        record(status, "Group overlap", len(overlap), action)
        print("Overlapping groups:", len(overlap), "| Forbidden:", GROUP_OVERLAP_FORBIDDEN)

    if TIME_COLUMN and TIME_COLUMN in train_df and TIME_COLUMN in valid_df:
        train_time = pd.to_datetime(train_df[TIME_COLUMN], errors="coerce")
        valid_time = pd.to_datetime(valid_df[TIME_COLUMN], errors="coerce")
        train_max = train_time.max()
        valid_min = valid_time.min()
        chronological = pd.isna(train_max) or pd.isna(valid_min) or train_max < valid_min
        record("PASS" if chronological else "WARN",
               "Chronological boundary",
               {"train_max": str(train_max), "validation_min": str(valid_min)},
               "If predicting the future, ensure training ends before validation begins.")
        print("Train max time:", train_max)
        print("Validation min time:", valid_min)

train_df = valid_df = None

if TRAIN_PATH and VALID_PATH and TRAIN_PATH.exists() and VALID_PATH.exists():
    train_df = pd.read_csv(TRAIN_PATH)
    valid_df = pd.read_csv(VALID_PATH)
elif SPLIT_COLUMN and SPLIT_COLUMN in df.columns:
    labels = df[SPLIT_COLUMN].astype(str).str.lower()
    train_df = df[labels.eq("train")].copy()
    valid_df = df[labels.isin(["validation", "valid", "val", "test"])].copy()

if train_df is not None and valid_df is not None and len(train_df) and len(valid_df):
    check_split_frames(train_df, valid_df)
else:
    record("MANUAL", "Split verification", "No train/validation files or split column supplied",
           "Provide split evidence: sizes plus class ratios, date ranges, or zero group overlap.")
    print("No split frames supplied. Complete the manual split evidence in docs/data_audit.md.")


## Step 5 — Project-type checks

In [ ]:
if PROJECT_TYPE == "nlp":
    if not TEXT_COLUMN or TEXT_COLUMN not in df.columns:
        record("WARN", "NLP text column", TEXT_COLUMN, "Set TEXT_COLUMN.")
    else:
        text = df[TEXT_COLUMN].fillna("").astype(str).str.strip()
        empty = int(text.eq("").sum())
        exact_dup = int(text.duplicated().sum())
        record("PASS" if empty == 0 else "WARN", "Empty text", empty, "Remove or justify empty text.")
        record("PASS" if exact_dup == 0 else "WARN", "Exact text duplicates", exact_dup,
               "Control duplicates before splitting and check conflicting labels.")
        print("Empty text:", empty, "| Exact duplicate text:", exact_dup)

elif PROJECT_TYPE == "cv":
    record("MANUAL", "CV manifest", "Expected columns: filepath, label, subject/group, split",
           "Check corrupted files, duplicate hashes, subject overlap, and training-only augmentation.")

elif PROJECT_TYPE == "unsupervised":
    numeric = df.select_dtypes(include=np.number)
    if numeric.shape[1] >= 2:
        scales = numeric.std(numeric_only=True).replace(0, np.nan).dropna()
        ratio = float(scales.max() / scales.min()) if len(scales) else np.nan
        record("WARN" if pd.notna(ratio) and ratio > 100 else "PASS",
               "Numeric scale difference", ratio,
               "Use a documented reusable scaler when distance/variance-based methods require it.")
        display(scales.sort_values(ascending=False).to_frame("std"))
    record("MANUAL", "Unsupervised evaluation plan", "stability + usefulness + internal metrics",
           "Do not invent a target; state how the result will be evaluated.")

else:
    record("MANUAL", "Feature availability at prediction time", "Human review required",
           "Mark post-outcome fields, future information, and target proxies in the leakage table.")


## Step 6 — Preprocessing manifest and fit-boundary evidence

In [ ]:
manifest_path = PROJECT_ROOT / "reports" / "preprocessing_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    display(manifest)
    fit_boundary = str(manifest.get("fit_boundary", "")).lower()
    if "train" in fit_boundary:
        record("PASS", "Preprocessing fit boundary", manifest.get("fit_boundary"))
    else:
        record("WARN", "Preprocessing fit boundary", manifest.get("fit_boundary"),
               "State explicitly that learned preprocessing is fitted on training data only.")
else:
    record("WARN", "Preprocessing manifest", manifest_path,
           "Create reports/preprocessing_manifest.json or link equivalent evidence in docs/data_audit.md.")


## Step 7 — Final report and suggested gate status

In [ ]:
report = pd.DataFrame(results)
order = {"FAIL": 0, "WARN": 1, "MANUAL": 2, "PASS": 3}
report["_order"] = report["status"].map(order)
report = report.sort_values(["_order", "check"]).drop(columns="_order").reset_index(drop=True)
display(report)

if (report["status"] == "FAIL").any():
    suggested_status = "RED"
elif report["status"].isin(["WARN", "MANUAL"]).any():
    suggested_status = "YELLOW"
else:
    suggested_status = "GREEN"

status_counts = report["status"].value_counts().to_dict()
print("Status counts:", status_counts)
print("Suggested Data Gate status:", suggested_status)

out_dir = PROJECT_ROOT / "reports" / "data_gate_self_check"
out_dir.mkdir(parents=True, exist_ok=True)
report.to_csv(out_dir / "data_gate_self_check.csv", index=False)

lines = [
    "# Data Gate Self-Check",
    "",
    f"- **Suggested status:** {suggested_status}",
    f"- **Project type:** {PROJECT_TYPE}",
    f"- **Dataset:** {DATASET_PATH}",
    "",
    "| Status | Check | Evidence | Required action |",
    "|---|---|---|---|",
]
for row in report.itertuples(index=False):
    evidence = str(row.evidence).replace("|", "/").replace("\n", " ")
    action = str(row.required_action).replace("|", "/").replace("\n", " ")
    lines.append(f"| {row.status} | {row.check} | {evidence} | {action} |")
(out_dir / "data_gate_self_check.md").write_text("\n".join(lines), encoding="utf-8")

print("Saved:", out_dir / "data_gate_self_check.csv")
print("Saved:", out_dir / "data_gate_self_check.md")


## Manual confirmation before the milestone commit

- [ ] I reviewed every feature for real prediction-time availability.
- [ ] I confirmed that no validation/test data taught preprocessing statistics, categories, vocabulary, thresholds, or feature selection.
- [ ] I opened every evidence path listed in `PROJECT_STATUS.md`.
- [ ] Any Yellow correction has an owner and due point.
- [ ] I checked `git status` and the staged diff before committing.
- [ ] I did not start REST/Flask; that belongs to C5.

Suggested milestone:

```bash
git commit -m "data: complete Data Gate audit and leakage-safe pipeline"
```
